# Part 5: Integrated Waste Management Assistant

One assistant that takes a photo, a description, or both, and returns the waste category, recycling instructions and the policy text they came from. It joins the three models:

- the CNN from Part 2 (`waste_cnn.keras`),
- the text classifier from Part 3 (rebuilt here in seconds),
- the instruction generator from Part 4 (the folder `waste_rag_model`).

Needs next to this notebook: `waste_cnn.keras`, the folder `waste_rag_model`, the folder `RealWaste`, `waste_descriptions.csv` and `waste_policy_documents.json`. TensorFlow and PyTorch are both used. Close other notebooks first, because both models sit in memory. Numbers in the Results come from my runs.

## Step 1: Loading Libraries

In [ ]:
import os
os.environ["USE_TF"] = "0"          # keep transformers from loading TensorFlow as well
import re
import json
import time
import uuid
import datetime
import tempfile
import numpy as np
import pandas as pd
import torch                         # imported before TensorFlow (the order that works on this machine)
import tensorflow as tf
from tensorflow import keras
from PIL import Image

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score

np.random.seed(42)

## Step 2: Check the Files and Load the Saved Models

In [ ]:
for name in ["waste_cnn.keras", "waste_rag_model", "RealWaste", "waste_descriptions.csv", "waste_policy_documents.json"]:
    print(name, "->", "found" if os.path.exists(name) else "MISSING")

cnn = keras.models.load_model("waste_cnn.keras", compile=False)
print("CNN output shape:", cnn.output_shape)

tok = AutoTokenizer.from_pretrained("waste_rag_model")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("waste_rag_model").float().eval()
print("generator parameters:", sum(p.numel() for p in gen_model.parameters()))

**Result:** All five files are found. The CNN outputs 9 probabilities and the generator has 76,961,152 parameters. PyTorch and TensorFlow load together in one process. transformers warns that two weight tables in the saved config differ and will not be tied; the Glass check in Step 8 shows the reload kept the fine-tuned behaviour.

## Step 3: The Held-Out Images

The 950 images the CNN never trained on, rebuilt with the same split settings as Part 1. The file path holds the class folder, so the true label can be read from it. We read images from these paths, not from the dataset's iteration order (that order caused the overlap bug in Part 1).

In [ ]:
holdout = keras.utils.image_dataset_from_directory(
    "RealWaste", validation_split=0.2, subset="validation", seed=42,
    image_size=(224, 224), batch_size=32, label_mode="categorical", shuffle=True)
paths = holdout.file_paths
print("hold-out images:", len(paths))
print(paths[:3])

**Result:** 950 images, matching Part 1. About half of them were the validation set that chose the tuned CNN, so scores on them are slightly optimistic. The exact 475 test images of Part 2 cannot be recovered, because their file names were not kept.

## Step 4: Policy Chunks

Same corpus as Part 4: category documents, one copy of the general rules, and one disposal card per category.

In [ ]:
desc = pd.read_csv("waste_descriptions.csv")
policies = json.load(open("waste_policy_documents.json"))
header_pattern = re.compile(r"^([A-Za-z][A-Za-z \-/&]+):\s*$", re.M)

def split_sections(text):
    parts = header_pattern.split(text)
    return list(zip(parts[1::2], [b.strip() for b in parts[2::2]]))

chunks = []
for policy in policies:
    for header, body in split_sections(policy["document_text"]):
        categories = policy["categories_covered"]
        match = re.match(r"([A-Z ]+) GUIDELINES$", header)
        if len(categories) > 1:
            if header == "GENERAL REQUIREMENTS":
                categories = ["ALL"]
            elif match:
                own = [c for c in categories if c.upper() == match.group(1).strip()]
                categories = own or categories
        chunks.append({"chunk_id": f"policy{policy['policy_id']}::{header}", "policy_id": policy["policy_id"],
                       "header": header, "category": categories[0],
                       "text": f"{policy['policy_type']} | {header}\n{body}"})
chunk_table = pd.DataFrame(chunks)

In [ ]:
own_docs = chunk_table[chunk_table["policy_id"] <= 9]
general_chunk = chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"].head(1)

cards = []
for category, group in desc.groupby("category"):
    instructions = sorted(group["disposal_instruction"].dropna().unique())
    note = group["common_confusion"].dropna().unique()
    text = (f"{category} | Disposal card\nComposition: {group['material_composition'].iloc[0]}\n"
            f"Disposal: " + " ".join(instructions))
    if len(note) > 0:
        text += "\nCommon confusion: " + note[0]
    cards.append({"chunk_id": f"card::{category}", "policy_id": None, "header": "Disposal card",
                  "category": category, "text": text})

rag_corpus = pd.concat([own_docs, general_chunk, pd.DataFrame(cards)], ignore_index=True)
CATEGORIES = sorted(desc["category"].unique())
general_row = rag_corpus[rag_corpus["category"] == "ALL"].iloc[0]
print("chunks:", len(rag_corpus))

## Step 5: Retrieval

Hybrid search (TF-IDF weight 0.25 plus MiniLM embeddings), restricted to the requested category and the general chunk.

In [ ]:
chunk_category = rag_corpus["category"].values
tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
tfidf_matrix = tfidf.fit_transform(rag_corpus["text"].tolist())
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
dense_matrix = encoder.encode(rag_corpus["text"].tolist(), normalize_embeddings=True, show_progress_bar=False)

def hybrid_scores(question, alpha=0.25):
    a = cosine_similarity(tfidf.transform([question]), tfidf_matrix)[0]
    b = dense_matrix @ encoder.encode([question], normalize_embeddings=True)[0]
    a = (a - a.mean()) / (a.std() + 1e-9)
    b = (b - b.mean()) / (b.std() + 1e-9)
    return alpha * a + (1 - alpha) * b

def get_chunks(question, category, k=6):
    scores = hybrid_scores(question)
    scores = np.where(np.isin(chunk_category, [category, "ALL"]), scores, -np.inf)
    best = np.argsort(-scores)[:k]
    return [rag_corpus.iloc[i] for i in best]

## Step 6: The Instruction Generator and Its Checker

The same functions as Part 4: build the answer for a category, check it section by section, rebuild it from the sources if the check fails.

In [ ]:
def chunk_text(category, header):
    row = rag_corpus[(rag_corpus["category"] == category) & (rag_corpus["header"] == header)]
    if row.empty:
        return ""
    return row["text"].iloc[0].split("\n", 1)[1].strip()

def bullets_of(body):
    return [line[2:].strip() for line in body.splitlines() if line.startswith("- ")]

def prose_of(body):
    lines = [line.strip() for line in body.splitlines() if line.strip() and not line.startswith("- ")]
    return " ".join(lines)

def reference_answer(category):
    card = chunk_text(category, "Disposal card")
    disposal = card.split("Disposal: ")[1].split("\nCommon confusion:")[0].strip()
    note = ""
    if "Common confusion: " in card:
        note = card.split("Common confusion: ")[1].strip()
    if category == "Miscellaneous Trash":
        cannot = bullets_of(chunk_text(category, "Items That Cannot Be Recycled"))
        special = bullets_of(chunk_text(category, "Special Handling Items"))
        text = (f"{category} cannot be recycled as ordinary material. Cannot be recycled: {'; '.join(cannot)}. "
                f"Special handling: {'; '.join(special)}. {disposal}")
    else:
        collect = prose_of(chunk_text(category, "Collection Method"))
        prepare = bullets_of(chunk_text(category, "Preparation Instructions"))
        avoid = bullets_of(chunk_text(category, "Non-Acceptable Items"))
        text = (f"To recycle {category}: {collect} Before recycling: {'; '.join(prepare)}. "
                f"Do not recycle: {'; '.join(avoid)}. {disposal}")
    if note:
        text += " Note: " + note
    return text.strip()

references = {c: reference_answer(c) for c in CATEGORIES}

In [ ]:
def make_input(question, chunk_rows):
    context = " ".join(" ".join(row["text"].split()) for row in chunk_rows)
    return f"Answer the question using only the context. Question: {question} Context: {context}"

def generate_answer(question, chunk_rows):
    inputs = tok(make_input(question, chunk_rows), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=200, num_beams=1)
    return tok.decode(output[0], skip_special_tokens=True)

In [ ]:
HEADINGS = ["Before recycling:", "Do not recycle:", "Cannot be recycled:", "Special handling:", "Note:"]
SOURCE_HEADING = {"Preparation Instructions": "Before recycling:", "Non-Acceptable Items": "Do not recycle:",
                  "Items That Cannot Be Recycled": "Cannot be recycled:", "Special Handling Items": "Special handling:"}

def check_answer(answer, chunk_rows):
    problems = []
    for heading in HEADINGS:                                       # rule 1: no heading twice
        if answer.count(heading) > 1:
            problems.append(f"heading repeated: {heading}")

    positions = sorted((m.start(), h) for h in HEADINGS for m in re.finditer(re.escape(h), answer))
    def heading_before(index):
        earlier = [h for p, h in positions if p < index]
        return earlier[-1] if earlier else None

    for row in chunk_rows:                                         # rule 2: bullets under their own heading
        expected = SOURCE_HEADING.get(row["header"])
        for bullet in bullets_of(row["text"].split("\n", 1)[1]):
            for match in re.finditer(re.escape(bullet), answer):
                found_under = heading_before(match.start())
                if expected is None:
                    problems.append(f"'{bullet}' (from {row['header']}) should not appear")
                elif found_under != expected:
                    problems.append(f"'{bullet}' is under '{found_under}', expected '{expected}'")

    for heading in HEADINGS[:4]:                                   # rule 3: heading needs its source chunk
        has_source = any(SOURCE_HEADING.get(row["header"]) == heading for row in chunk_rows)
        if heading in answer and not has_source:
            problems.append(f"{heading} written but its source chunk was not retrieved")
    return problems

In [ ]:
answer_cache = {}          # the answer depends only on the category, so we keep it

def generate_instructions(category, question=None, k=6):
    matches = [c for c in CATEGORIES if c.lower() == str(category).strip().lower()]
    if not matches:
        return {"category": category, "answer": None, "status": "unknown category", "problems": [], "sources": []}
    category = matches[0]
    question = question or f"How to recycle {category}"
    key = (category, question, k)
    if key in answer_cache:
        return answer_cache[key]

    chunk_rows = get_chunks(question, category, k)
    raw = generate_answer(question, chunk_rows)
    problems = check_answer(raw, chunk_rows)
    if problems:
        answer, status = reference_answer(category), "fallback (failed the check; rebuilt from the sources)"
    else:
        answer, status = raw, "ok"
    result = {"category": category, "answer": answer, "status": status, "problems": problems,
              "sources": [{"chunk_id": row["chunk_id"], "text": row["text"]} for row in chunk_rows]}
    answer_cache[key] = result
    return result

## Step 7: The Text Classifier

Rebuilt from the CSV with the Part 3 settings: same cleaning, same split and seed, TF-IDF with word pairs and a calibrated LinearSVC, fitted on training and validation descriptions only.

In [ ]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_data = desc[["description", "category"]].copy()
text_data["clean"] = text_data["description"].apply(clean_text)
text_data = text_data.drop_duplicates(subset="clean").reset_index(drop=True)
LABELS = sorted(text_data["category"].unique())
label_to_id = {label: i for i, label in enumerate(LABELS)}
text_data["y"] = text_data["category"].map(label_to_id)

train_data, hold_data = train_test_split(text_data, test_size=0.20, stratify=text_data["y"], random_state=42)
val_data, test_data = train_test_split(hold_data, test_size=0.50, stratify=hold_data["y"], random_state=42)

dev_text = pd.concat([train_data["clean"], val_data["clean"]])
dev_y = pd.concat([train_data["y"], val_data["y"]])

def new_text_model():
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                     ("clf", CalibratedClassifierCV(LinearSVC(), cv=5))])

text_model = new_text_model()
text_model.fit(dev_text, dev_y)
print("text model test accuracy:", round(accuracy_score(test_data["y"], text_model.predict(test_data["clean"])), 4))

In [ ]:
CONFIDENCE_THRESHOLD = 0.40

def predict_waste_category(text, threshold=CONFIDENCE_THRESHOLD):
    if not isinstance(text, str) or clean_text(text) == "":
        return {"category": None, "confidence": 0.0, "status": "empty input", "top3": []}
    cleaned = clean_text(text)
    word_numbers = text_model.named_steps["tfidf"].transform([cleaned])
    if word_numbers.nnz == 0:
        return {"category": None, "confidence": 0.0, "status": "no known words", "top3": []}
    probs = text_model.predict_proba([cleaned])[0]
    order = np.argsort(-probs)[:3]
    top3 = [(LABELS[i], round(float(probs[i]), 2)) for i in order]
    status = "ok" if probs[order[0]] >= threshold else "low confidence"
    return {"category": LABELS[order[0]], "confidence": round(float(probs[order[0]]), 2),
            "status": status, "top3": top3}

## Step 8: Smoke Test of the Three Models

In [ ]:
def load_for_cnn(path):
    image = keras.utils.load_img(path, target_size=(224, 224))
    return keras.utils.img_to_array(image)[None, ...]        # shape (1, 224, 224, 3), values 0-255

true_label = os.path.basename(os.path.dirname(paths[0]))
probs = cnn.predict(load_for_cnn(paths[0]), verbose=0)[0]
print("image:", paths[0], "| true:", true_label, "| CNN says:", CATEGORIES[int(probs.argmax())], round(float(probs.max()), 2))

print("text:", predict_waste_category("red glass pen"))

glass = generate_instructions("Glass")
print("generator status:", glass["status"], "| matches the Part 4 reference:", glass["answer"] == references["Glass"])
print("sources:", [s["chunk_id"] for s in glass["sources"]])

**Result:** Text model test accuracy 1.0, as in Part 3; "red glass pen" (wrong in two earlier models) is Miscellaneous Trash at 0.91. The first hold-out image (truly Plastic) is classified Plastic at confidence 0.65. The reloaded generator writes the Glass answer identical to the Part 4 reference, using the six Glass chunks as sources (the general chunk is added by the assistant in Step 11).

## Step 9: How Reliable Is the CNN's Confidence?

We run the CNN over all 950 hold-out images (reading images from their paths), and compare its confidence with whether it was right.

In [ ]:
def cnn_probabilities(path_list, batch_size=32):
    batches = []
    for i in range(0, len(path_list), batch_size):
        images = np.concatenate([load_for_cnn(p) for p in path_list[i:i + batch_size]])
        batches.append(cnn.predict(images, verbose=0))
    return np.concatenate(batches)

true_numbers = np.array([CATEGORIES.index(os.path.basename(os.path.dirname(p))) for p in paths])
start = time.time()
cnn_probs = cnn_probabilities(paths)
seconds = time.time() - start
predicted = cnn_probs.argmax(1)
confidence = cnn_probs.max(1)
print("seconds:", round(seconds), "| per image:", round(seconds / len(paths), 3))
print("hold-out accuracy:", round(float((predicted == true_numbers).mean()), 4))

rows = []
for low, high in [(0.0, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]:
    in_band = (confidence >= low) & (confidence < high)
    rows.append({"confidence band": f"{low:.1f}-{min(high, 1):.1f}", "images": int(in_band.sum()),
                 "accuracy": round(float((predicted[in_band] == true_numbers[in_band]).mean()), 3)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# pictures that are not waste
random_state = np.random.RandomState(0)
synthetic = {
    "black": np.zeros((1, 224, 224, 3), "float32"),
    "white": np.full((1, 224, 224, 3), 255, "float32"),
    "random noise": random_state.randint(0, 256, (1, 224, 224, 3)).astype("float32"),
    "gradient": np.tile(np.linspace(0, 255, 224, dtype="float32")[None, None, :, None], (1, 224, 1, 3)),
}
for name, image in synthetic.items():
    probs = cnn.predict(image, verbose=0)[0]
    print(f"{name:14} -> {CATEGORIES[int(probs.argmax())]:20} confidence {probs.max():.2f}")

**Result:** 30 seconds for 950 images (0.032 s each). Hold-out accuracy 0.800. By confidence band: 0.0-0.5: 0.514 (70 images); 0.5-0.6: 0.543 (105); 0.6-0.7: 0.554 (83); 0.7-0.8: 0.690 (100); 0.8-1.0: 0.932 (592). Below 0.7 the CNN is close to a coin flip, with steps up at 0.7 and 0.8. Non-waste pictures: black gets Metal at 0.45, white Cardboard at 0.28, noise Plastic at 0.57 (0.54 after resizing in the assistant), gradient Cardboard at 0.51, so the CNN cannot reject non-waste images itself. Decision, cut-offs 0.80 and 0.70: at least 0.80 answer directly (62% of images, 93% accurate); 0.70 to 0.80 answer and ask to confirm (10.5%, 69%); below 0.70 (27%) ask for a description or a clearer photo. A first guess of 0.50 and 0.70 was dropped because noise at 0.54 was still answered and 0.5 to 0.7 is no more reliable than below. The cut-offs were set on the same 950 images and each low band has only 70 to 105 images, so they are rough.

## Step 10: The Assistant

`assist(image_path, text)` checks each input, classifies the photo and/or the description, and decides: one usable input gives an answer; agreeing inputs give an answer; disagreeing inputs give a conflict (the user confirms, instructions for both are shown); nothing usable gives "need more information".

In [ ]:
IMG_CONFIDENT, IMG_UNCERTAIN = 0.80, 0.70
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
MAX_TEXT_CHARS = 1000

def classify_image(path):
    start = time.time()
    empty = {"category": None, "confidence": 0.0, "top3": [], "seconds": 0.0}
    try:
        path = str(path)
        if not os.path.exists(path):
            return {**empty, "status": "error: file not found"}
        if os.path.splitext(path)[1].lower() not in IMAGE_EXTENSIONS:
            return {**empty, "status": "error: unsupported file type (use .jpg or .png)"}
        probs = cnn.predict(load_for_cnn(path), verbose=0)[0]
    except Exception as error:                              # unreadable or corrupt file
        return {**empty, "status": f"error: cannot read the image ({type(error).__name__})"}
    order = np.argsort(-probs)[:3]
    top = float(probs[order[0]])
    level = "confident" if top >= IMG_CONFIDENT else "uncertain" if top >= IMG_UNCERTAIN else "unsure"
    return {"status": level, "category": CATEGORIES[order[0]], "confidence": round(top, 2),
            "top3": [(CATEGORIES[i], round(float(probs[i]), 2)) for i in order], "seconds": round(time.time() - start, 3)}

In [ ]:
def classify_text(text):
    start = time.time()
    if not isinstance(text, str):
        return {"category": None, "confidence": 0.0, "top3": [], "status": "error: the description must be text", "seconds": 0.0}
    result = predict_waste_category(text[:MAX_TEXT_CHARS])
    result["seconds"] = round(time.time() - start, 3)
    return result

def usable_guesses(image_result, text_result):
    guesses = []
    if image_result and image_result["status"] in ("confident", "uncertain"):
        guesses.append({"source": "image", "category": image_result["category"], "level": image_result["status"]})
    if text_result and text_result["status"] == "ok":
        guesses.append({"source": "text", "category": text_result["category"], "level": "confident"})
    return guesses

In [ ]:
def need_more_result(image_result, text_result):
    reasons, options = [], []
    if image_result:
        reasons.append("photo: " + image_result["status"])
        options = image_result["top3"]
    if text_result:
        reasons.append("description: " + text_result["status"])
        options = options or text_result["top3"]
    message = "I could not tell what this is (" + "; ".join(reasons) + "). Please add a short description or a clearer photo."
    return {"status": "need more information", "needs_confirmation": True, "options": options, "message": message}

def conflict_result(guesses):
    message = (f"The photo suggests {guesses[0]['category']} but the description suggests {guesses[1]['category']}. "
               "Please confirm which one it is. Instructions for both are shown.")
    return {"status": "conflict", "needs_confirmation": True, "message": message}

def answered_result(category, guesses, image_result, text_result):
    uncertain = len(guesses) == 1 and guesses[0]["level"] == "uncertain"
    used = " and ".join(g["source"] for g in guesses)
    message = f"Based on the {used}: {category}."
    if uncertain:
        message += " I am not fully sure, please check."
    if image_result and image_result["status"] not in ("confident", "uncertain"):
        message += " (the photo was too unclear to use)"
    if text_result and text_result["status"] != "ok":
        message += " (the description was too unclear to use)"
    status = "answered, please confirm" if uncertain else "answered"
    return {"status": status, "needs_confirmation": uncertain, "category": category, "message": message}

def add_instructions(result, categories):
    for category in categories:
        answer = generate_instructions(category)
        result["instructions"].append({"category": category, "answer": answer["answer"],
                                       "status": answer["status"], "problems": answer["problems"]})
        known = [s["chunk_id"] for s in result["sources"]]
        result["sources"] += [s for s in answer["sources"] if s["chunk_id"] not in known]
    result["sources"].append({"chunk_id": general_row["chunk_id"], "text": general_row["text"]})

In [ ]:
request_log = {}

def assist(image_path=None, text=None):
    start = time.time()
    request_id = uuid.uuid4().hex[:8]
    has_image = image_path not in (None, "")
    has_text = text is not None and not (isinstance(text, str) and text.strip() == "")
    image_result = classify_image(image_path) if has_image else None
    text_result = classify_text(text) if has_text else None
    guesses = usable_guesses(image_result, text_result)

    result = {"request_id": request_id, "image": image_result, "text": text_result, "instructions": [],
              "sources": [], "needs_confirmation": False, "input_text": text if isinstance(text, str) else None}

    if image_result is None and text_result is None:
        result.update(status="no input", message="Please give me a photo or a short description of the item.")
    elif len(guesses) == 0:
        result.update(need_more_result(image_result, text_result))
    else:
        categories = sorted({g["category"] for g in guesses})
        if len(categories) > 1:
            result.update(conflict_result(guesses))
        else:
            result.update(answered_result(categories[0], guesses, image_result, text_result))
        add_instructions(result, categories)

    result["seconds"] = round(time.time() - start, 3)
    request_log[request_id] = result
    return result

def show(result):
    print(f"[{result['request_id']}] {result['status']} ({result['seconds']} s) - {result['message']}")
    for item in result["instructions"]:
        print(f"  {item['category']} [{item['status']}]: {item['answer'][:110]}...")
    if result["sources"]:
        print("  policy sources:", [s["chunk_id"] for s in result["sources"]])

## Step 11: Edge Cases, Speed and a Real Test Image

The answer for each category is generated once and cached, so we warm the cache first and time it. Then cases that should not crash it.

In [ ]:
def first_path(category):
    for p in paths:
        if os.path.basename(os.path.dirname(p)) == category:
            return p

start = time.time()
for category in CATEGORIES:
    generate_instructions(category)
print("warm-up seconds (9 categories):", round(time.time() - start))

cases = [
    ("description only", dict(text="empty green glass bottle with contents")),
    ("no input", dict()),
    ("whitespace text", dict(text="   ")),
    ("missing file", dict(image_path="does_not_exist.jpg")),
    ("wrong file type", dict(image_path="waste_descriptions.csv")),
    ("gibberish", dict(text="asdf qwerty")),
    ("other language", dict(text="chupa ya plastiki")),
    ("text that is not a string", dict(text=123)),
]
rows = []
for name, arguments in cases:
    answer = assist(**arguments)
    rows.append({"case": name, "status": answer["status"], "category": answer.get("category"), "seconds": answer["seconds"]})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
with tempfile.TemporaryDirectory() as folder:                  # temporary files, deleted automatically
    black_path = os.path.join(folder, "black.png")
    Image.new("RGB", (300, 300), (0, 0, 0)).save(black_path)
    noise_path = os.path.join(folder, "noise.png")
    noise = np.random.RandomState(0).randint(0, 256, (300, 300, 3), dtype="uint8")
    Image.fromarray(noise).save(noise_path)
    for name, p in [("black image", black_path), ("noise image", noise_path)]:
        answer = assist(image_path=p)
        print(name, "->", answer["status"], "| CNN confidence", answer["image"]["confidence"])

for name, p in [("Plastic hold-out", first_path("Plastic")), ("Glass hold-out", first_path("Glass"))]:
    answer = assist(image_path=p)
    print(name, "->", answer["status"], "| CNN confidence", answer["image"]["confidence"])

# a description rescues an unclear photo
answer = assist(image_path=first_path("Plastic"), text="plastic bottle")
print("\nPlastic photo + 'plastic bottle':", answer["status"], "|", answer["message"])
answer = assist(image_path=first_path("Plastic"), text="crushed aluminium can")
print("Plastic photo + 'crushed aluminium can':", answer["status"], "|", answer["message"])
show(assist(text="glass jar"))

**Result:** Warm-up took 32 seconds for 9 answers (about 3.6 s each); after that an image request takes about 0.12 to 0.15 s (mostly the CNN) and a description-only request about 0.01 s. No input and whitespace give "no input"; a missing file, a CSV passed as an image, gibberish, a Swahili phrase and a non-text input all give "need more information" with a reason; nothing crashed. Black (CNN 0.45), noise (0.54) and both real hold-out images (Plastic 0.65, Glass 0.54, both classified correctly) fall below 0.70 and ask for more, so two correct guesses are lost: that is the price of the cut-off. A usable description takes over, and the message says the photo was too unclear to use ("plastic bottle" gives Plastic; "crushed aluminium can" gives Metal). Trusting the description when the photo is unsure is reasonable (the photo is right only about 54% of the time there), but the text model has only been tested on template-style descriptions, and a real description with typos could be read wrongly with nothing to check it against.

## Step 12: End-to-End Test: 360 Requests

90 hold-out images (10 per category, random) in four situations: image only; with an agreeing description (a random Part 3 test description of the same category); with a conflicting description (one from another category); with gibberish.

In [ ]:
random_state = np.random.RandomState(0)
by_class = {c: [p for p in paths if os.path.basename(os.path.dirname(p)) == c] for c in CATEGORIES}
sample = [(c, p) for c in CATEGORIES for p in random_state.choice(by_class[c], 10, replace=False)]
description_pool = {c: test_data[test_data["category"] == c]["description"].tolist() for c in CATEGORIES}

rows = []
for true_category, image_path in sample:
    other = random_state.choice([c for c in CATEGORIES if c != true_category])
    agreeing = random_state.choice(description_pool[true_category])
    conflicting = random_state.choice(description_pool[other])
    for kind, text in [("image only", None), ("agreeing description", agreeing),
                       ("conflicting description", conflicting), ("gibberish description", "asdf qwerty")]:
        answer = assist(image_path=image_path, text=text)
        rows.append({"true": true_category, "kind": kind, "status": answer["status"],
                     "category": answer.get("category"), "seconds": answer["seconds"]})
test_results = pd.DataFrame(rows)

In [ ]:
rows = []
for kind, group in test_results.groupby("kind"):
    answered = group[group["status"].str.startswith("answered")]
    correct = None
    if kind != "conflicting description" and len(answered) > 0:
        correct = round(float((answered["category"] == answered["true"]).mean()), 3)
    rows.append({"kind": kind, "requests": len(group), "answered": len(answered),
                 "conflict": int((group["status"] == "conflict").sum()),
                 "need more info": int((group["status"] == "need more information").sum()),
                 "correct when answered": correct, "median seconds": round(float(group["seconds"].median()), 3)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** Image only: 74 of 90 answered, 16 sent back, 0.905 correct (about 7 wrong). Gibberish description: identical, so useless text changes nothing. Agreeing description: 83 answered, all correct, 7 conflicts: the 16 refused photos are answered from the description, and the 7 conflicts equal the 7 wrong image-only answers (the counts match; the images were not compared one by one), so every confident CNN error became a request to confirm. Conflicting description: 73 of 90 flagged (73 of the 74 with a usable photo); the 16 with an unusable photo are answered silently from the text, with a note. Median about 0.11 s. One random sample of 90; the descriptions are template-style, so this shows the combination logic, not accuracy on real user text.

## Step 13: Feedback

`give_feedback` records whether an answer helped and, optionally, the correct category (unknown request ids and categories are rejected). `learn_from_feedback` adds corrected descriptions to the training data and refits the text model, accepting the new model only if test accuracy does not fall. Image feedback is only logged, because retraining the CNN is too slow for a feedback loop.

In [ ]:
feedback_log = []

def give_feedback(request_id, helpful=None, correct_category=None, comment=""):
    if request_id not in request_log:
        return "unknown request id"
    if correct_category is not None:
        matches = [c for c in CATEGORIES if c.lower() == str(correct_category).strip().lower()]
        if not matches:
            return f"unknown category (choose from {CATEGORIES})"
        correct_category = matches[0]
    feedback_log.append({"request_id": request_id, "helpful": helpful, "correct_category": correct_category,
                         "comment": comment, "time": datetime.datetime.now().isoformat(timespec="seconds")})
    return "thank you, feedback recorded"

In [ ]:
def learn_from_feedback():
    global text_model
    corrections = []
    for entry in feedback_log:
        text = request_log[entry["request_id"]].get("input_text")
        if entry["correct_category"] and text:
            corrections.append((text, entry["correct_category"]))
    if not corrections:
        return "no usable corrections yet"

    new_text = pd.Series([clean_text(t) for t, c in corrections])
    new_y = pd.Series([label_to_id[c] for t, c in corrections])
    candidate = new_text_model()
    candidate.fit(pd.concat([dev_text, new_text], ignore_index=True), pd.concat([dev_y, new_y], ignore_index=True))

    old_accuracy = accuracy_score(test_data["y"], text_model.predict(test_data["clean"]))
    new_accuracy = accuracy_score(test_data["y"], candidate.predict(test_data["clean"]))
    if new_accuracy >= old_accuracy:
        text_model = candidate
        return f"accepted {len(corrections)} correction(s): test accuracy {old_accuracy:.4f} -> {new_accuracy:.4f}"
    return f"rejected: test accuracy would fall {old_accuracy:.4f} -> {new_accuracy:.4f}"

In [ ]:
before = assist(text="chupa ya plastiki")
print("before:", before["status"], "|", before["message"])
print(give_feedback(before["request_id"], helpful=False, correct_category="plastic", comment="plastic bottle in Swahili"))
print(learn_from_feedback())

after = assist(text="chupa ya plastiki")
print("after:", after["status"], "|", after["message"])
related = assist(text="chupa ya plastiki tupu")
print("related wording:", related["status"], "|", related["message"])
print("unrelated cases:", assist(text="red glass pen")["category"], "|", assist(text="banana peel")["category"])
print(pd.DataFrame(feedback_log))
print(give_feedback("nope"), "|", give_feedback(before["request_id"], correct_category="furniture"))

**Result:** Before: the Swahili phrase "chupa ya plastiki" (plastic bottle) was refused (no known words). After one correction to Plastic the guard accepted the refit (test accuracy 1.0000 to 1.0000) and the phrase, and a related wording sharing its words, are answered as Plastic, while "red glass pen" (Miscellaneous Trash) and "banana peel" (Food Organics) keep their answers. Invalid feedback (unknown id, unknown category) is rejected. The guard cannot catch a wrong correction: in a private check, labelling the same phrase Glass was also accepted (test accuracy stayed 1.0) and the model then answered Glass at about 0.79, because no test description contains those words. Corrections need human review or agreement from several users. The log lives in memory and is lost when the notebook closes.

## Summary

- **Design:** photo and/or description, then input checks, the CNN and the text classifier, a decision step (agree, conflict, unsure), then the generator with its checker and an answer cache. The answer comes with the policy text it used and a request id for feedback.
- **Evidence:** every invalid input gave a clean message; the CNN cut-offs come from measured accuracy by confidence (93% above 0.8, about 54% below 0.7); in 360 requests an agreeing description turned all seven confident CNN errors into requests to confirm; a request takes about 0.11 s after a 32 s warm-up; one correction made a refused phrase answerable without harming other answers.
- **Limitations:** the CNN cannot recognise non-waste pictures (only synthetic ones tested); the text classifier and generator are tested only on template-style data; when a photo is unsure a description is trusted unchecked; the feedback guard cannot detect a wrong correction and the log is not saved; held-out images are partly the validation set; each experiment is a single run on a small sample.